# Cost Engine — Cost & Time Impact of Misclassification (Task Council)

This notebook computes **Severity of Misclassification** as a defensible, auditable score based on:

- **Financial impact** (salary delta between predicted vs plausible alternative role)
- **Operational impact** (time-to-fix constants, with worst-case rules)
- **Role distance** (adjacent vs cross-family vs vertical)

Optional: Use a **Task Council tandem**:
- **Primary (Sonnet 4.5)** = qualitative organizational risk tagger (does not do math)
- **Validator (DeepSeek-R1)** = audit-style checks & worst-case triggers

# ENHANCEMENTS IN THIS VERSION:
1. Benefit Load Factor (1.30) - Accounts for total compensation, not just salary
2. Asymmetric Risk - Underpayment has higher legal risk than overpayment
3. Updated time estimates (24 hours base, per MNPS validation)
4. Both 3-month and 6-month cost windows
5. Legal risk flagging for underpayment scenarios


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/Alternative_Role_Classifier_Analyzer.ipynb)

**📝 Before using:** Update the GitHub URL above with your actual username and repository name.

## **Enhanced Transformation Cell v2.0**

In [1]:
# ==============================================================================
# ENHANCED COST CALCULATOR v2.0
# Replaces Cells 3-13 with consolidated, production-ready implementation
# ==============================================================================
#
# ENHANCEMENTS:
# - Benefit Load Factor (1.30) - Total compensation, not just salary
# - Asymmetric Risk - Underpayment ×1.5, Overpayment ×0.75
# - Validated time estimates (24 hours base)
# - Dual cost windows (3-month, 6-month)
# - Legal risk flagging
# - Combined risk scoring (Likelihood × Severity)
# ==============================================================================

import pandas as pd
import numpy as np
from ast import literal_eval
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------------------------

# Time estimates (VALIDATED WITH MNPS HR)
BASE_HOURS = 24.0  # Average: 3 working days

# Financial assumptions
HR_LOADED_RATE = 65.0           # $/hour for HR labor
BENEFIT_LOAD_FACTOR = 1.30      # 30% benefits load
CORRECTION_WINDOW_MONTHS_SHORT = 3   # Conservative
CORRECTION_WINDOW_MONTHS_LONG = 6    # Moderate

# Asymmetric risk multipliers
UNDERPAYMENT_MULTIPLIER = 1.5   # Higher risk for underpaying
OVERPAYMENT_MULTIPLIER = 0.75   # Lower risk for overpaying

# Task Council Configuration (New)
USE_TASK_COUNCIL = False # Set to True to enable Task Council integration

print("=" * 80)
print("COST ACCOUNTANT - JOB MISCLASSIFICATION COST ANALYZER v2.0")
print("=" * 80)
print(f"\n📋 Configuration:")
print(f"   Base correction time: {BASE_HOURS} hours")
print(f"   HR loaded rate: ${HR_LOADED_RATE}/hour")
print(f"   Benefit load factor: {BENEFIT_LOAD_FACTOR}")
print(f"   Cost windows: {CORRECTION_WINDOW_MONTHS_SHORT}mo, {CORRECTION_WINDOW_MONTHS_LONG}mo")
print(f"   Asymmetric risk: Underpay ×{UNDERPAYMENT_MULTIPLIER}, Overpay ×{OVERPAYMENT_MULTIPLIER}")
print(f"   Task Council Enabled: {USE_TASK_COUNCIL}")

# ------------------------------------------------------------------------------
# 2. LOAD INPUT FILES
# ------------------------------------------------------------------------------

PATH_MASTER   = "/content/Master_Job_Analysis_claude_sonnet_4_5.csv"
PATH_SAMPLE   = "/content/Sample JDs.csv" # Corrected filename in previous turn
PATH_SALARY   = "/content/salary_by_major_role_grouping.csv"
PATH_TIME     = "/content/Time to correct an error in hours.csv" # Corrected filename

print(f"\n📂 Loading files...")

df_master = pd.read_csv(PATH_MASTER)
df_sample = pd.read_csv(PATH_SAMPLE, encoding='latin1')
df_salary = pd.read_csv(PATH_SALARY)
df_time   = pd.read_csv(PATH_TIME)

print(f"   ✅ Master: {len(df_master)} records")
print(f"   ✅ Sample JDs: {len(df_sample)} records")
print(f"   ✅ Salary: {len(df_salary)} roles")

# ------------------------------------------------------------------------------
# 3. LINK JOB CODES
# ------------------------------------------------------------------------------

print(f"\n🔗 Linking Job Codes...")

df_sample['title_clean'] = df_sample['Job Description Name'].str.lower().str.strip()
df_master['title_clean'] = df_master['job_title_original'].str.lower().str.strip()

df_master = df_master.merge(
    df_sample[['Job Code', 'title_clean']],
    on='title_clean',
    how='left'
)

matched = df_master['Job Code'].notna().sum()
print(f"   ✅ Matched {matched}/{len(df_master)} records")

# ------------------------------------------------------------------------------
# 4. CREATE SALARY MAP
# ------------------------------------------------------------------------------

print(f"\n💰 Creating salary map...")

df_salary.columns = [c.strip() for c in df_salary.columns]

def clean_salary(x):
    if pd.isna(x):
        return np.nan
    return float(str(x).replace('$', '').replace(',', '').strip())

df_salary['Average Annual Salary'] = df_salary['Average Annual Salary'].apply(clean_salary)
salary_map = df_salary.set_index('Major Role Grouping')['Average Annual Salary'].to_dict()

print(f"   ✅ Loaded {len(salary_map)} salary mappings")
print(f"   📊 Range: ${min(salary_map.values()):,.0f} - ${max(salary_map.values()):,.0f}")

# ------------------------------------------------------------------------------
# 5. EXPAND ALTERNATIVES INTO SCENARIOS
# ------------------------------------------------------------------------------

print(f"\n🔄 Expanding alternatives...")

scenarios = []

for idx, row in df_master.iterrows():
    job_code = row['Job Code'] if pd.notna(row['Job Code']) else f"UNK_{idx}"
    job_title = row['job_title_original']
    predicted = row['major_role_group']

    alt_list_str = row['alt_roles_list']
    if pd.isna(alt_list_str) or alt_list_str == '[]':
        scenarios.append({
            'Job Code': job_code,
            'Job Title': job_title,
            'Predicted Role': predicted,
            'Alternative Role': predicted,
            'Consensus': row.get('Consensus', 'YES'),
            'human_error_probability': row.get('human_error_probability'),
            'confusion_risk_score': row.get('confusion_risk_score'),
            'similarity_numeric': row.get('similarity_numeric'),
            'likelihood_error_score': row.get('likelihood_error_score_0_5'),
            'likelihood_band': row.get('likelihood_band'),
            'review_priority': row.get('review_priority'),
            'alt_count': 0
        })
        continue

    try:
        alternatives = literal_eval(alt_list_str)
    except:
        try:
            alternatives = [a.strip().strip("'\"") for a in alt_list_str.strip("[]").split(',')]
        except:
            continue

    for alt_role in alternatives:
        alt_role = str(alt_role).strip().strip("'\"")
        if alt_role:
            scenarios.append({
                'Job Code': job_code,
                'Job Title': job_title,
                'Predicted Role': predicted,
                'Alternative Role': alt_role,
                'Consensus': row.get('Consensus', 'YES'),
                'human_error_probability': row.get('human_error_probability'),
                'confusion_risk_score': row.get('confusion_risk_score'),
                'similarity_numeric': row.get('similarity_numeric'),
                'likelihood_error_score': row.get('likelihood_error_score_0_5'),
                'likelihood_band': row.get('likelihood_band'),
                'review_priority': row.get('review_priority'),
                'alt_count': row.get('alt_count', 0)
            })

df_scenarios = pd.DataFrame(scenarios)

print(f"   ✅ Created {len(df_scenarios)} scenarios from {len(df_master)} jobs")
print(f"   📊 Avg alternatives/job: {len(df_scenarios)/len(df_master):.1f}")

# ------------------------------------------------------------------------------
# 6. CALCULATE COSTS WITH ENHANCEMENTS
# ------------------------------------------------------------------------------

print(f"\n💵 Calculating costs...")

# Map salaries
df_scenarios['Pred_Salary'] = df_scenarios['Predicted Role'].map(salary_map)
df_scenarios['Alt_Salary'] = df_scenarios['Alternative Role'].map(salary_map)

# Calculate delta (preserve direction)
df_scenarios['Salary_Delta_Raw'] = df_scenarios['Pred_Salary'] - df_scenarios['Alt_Salary']
df_scenarios['Salary_Delta_Abs'] = df_scenarios['Salary_Delta_Raw'].abs()

# Flag underpayment (LEGAL RISK)
df_scenarios['Is_Underpaid'] = df_scenarios['Salary_Delta_Raw'] < 0
df_scenarios['Payment_Direction'] = df_scenarios['Is_Underpaid'].map({
    True: 'UNDERPAID',
    False: 'OVERPAID'
})

underpaid = df_scenarios['Is_Underpaid'].sum()
print(f"\n   📊 Underpaid: {underpaid} (HIGH legal risk)")
print(f"   📊 Overpaid:  {len(df_scenarios) - underpaid} (LOW risk)")

# Apply benefit load
df_scenarios['Total_Comp_Delta'] = df_scenarios['Salary_Delta_Abs'] * BENEFIT_LOAD_FACTOR

print(f"\n   💰 With {int((BENEFIT_LOAD_FACTOR-1)*100)}% benefits:")
print(f"      Avg increase: ${(df_scenarios['Total_Comp_Delta'] - df_scenarios['Salary_Delta_Abs']).mean():,.0f}")

# Calculate hours with multipliers
df_scenarios['Hours'] = BASE_HOURS
df_scenarios.loc[df_scenarios['Consensus'] == 'NO', 'Hours'] *= 1.25
df_scenarios.loc[df_scenarios['confusion_risk_score'] >= 2, 'Hours'] *= 1.15
df_scenarios.loc[df_scenarios['likelihood_error_score'] >= 4.0, 'Hours'] *= 1.15
df_scenarios.loc[df_scenarios['human_error_probability'] >= 24, 'Hours'] *= 1.10

# Apply asymmetric risk
df_scenarios['Risk_Multiplier'] = df_scenarios['Is_Underpaid'].map({
    True: UNDERPAYMENT_MULTIPLIER,
    False: OVERPAYMENT_MULTIPLIER
})

# Calculate costs for both windows
WINDOW_SHORT = CORRECTION_WINDOW_MONTHS_SHORT / 12
WINDOW_LONG = CORRECTION_WINDOW_MONTHS_LONG / 12

df_scenarios['Pay_Impact_3mo'] = df_scenarios['Total_Comp_Delta'] * WINDOW_SHORT * df_scenarios['Risk_Multiplier']
df_scenarios['Process_Cost'] = df_scenarios['Hours'] * HR_LOADED_RATE
df_scenarios['Total_Cost_3mo'] = df_scenarios['Pay_Impact_3mo'] + df_scenarios['Process_Cost']

df_scenarios['Pay_Impact_6mo'] = df_scenarios['Total_Comp_Delta'] * WINDOW_LONG * df_scenarios['Risk_Multiplier']
df_scenarios['Total_Cost_6mo'] = df_scenarios['Pay_Impact_6mo'] + df_scenarios['Process_Cost']

# Flag high-value underpayment
df_scenarios['Legal_Risk_Flag'] = df_scenarios['Is_Underpaid'] & (df_scenarios['Total_Cost_3mo'] > 5000)

# Keep only valid scenarios
df_scen = df_scenarios[df_scenarios['Total_Cost_3mo'].notna()].copy()

print(f"\n   ✅ Calculated costs for {len(df_scen)} valid scenarios")

print(f"\n📊 Cost Statistics:")
print(f"   3-Month Window:")
print(f"      Average: ${df_scen['Total_Cost_3mo'].mean():,.0f}")
print(f"      Max:     ${df_scen['Total_Cost_3mo'].max():,.0f}")
print(f"   6-Month Window:")
print(f"      Average: ${df_scen['Total_Cost_6mo'].mean():,.0f}")
print(f"      Max:     ${df_scen['Total_Cost_6mo'].max():,.0f}")

legal_count = df_scen['Legal_Risk_Flag'].sum()
print(f"\n   ⚠️  Legal risk scenarios: {legal_count}")

# ------------------------------------------------------------------------------
# 7. ASSIGN SEVERITY BANDS
# ------------------------------------------------------------------------------

print(f"\n📊 Assigning severity bands...")

p25 = df_scen['Total_Cost_3mo'].quantile(0.25)
p50 = df_scen['Total_Cost_3mo'].quantile(0.50)
p75 = df_scen['Total_Cost_3mo'].quantile(0.75)

def severity_band(cost):
    if pd.isna(cost):
        return 'Unknown'
    if cost < p25:
        return 'Low'
    elif cost < p50:
        return 'Moderate'
    elif cost < p75:
        return 'High'
    else:
        return 'Very High'

df_scen['Severity_Band'] = df_scen['Total_Cost_3mo'].apply(severity_band)

print(f"\n   Distribution:")
for band in ['Low', 'Moderate', 'High', 'Very High']:
    count = (df_scen['Severity_Band'] == band).sum()
    pct = count / len(df_scen) * 100
    print(f"      {band:12}: {count:3} ({pct:5.1f}%)")

print(f"\n   Thresholds:")
print(f"      Low:       < ${p25:,.0f}")
print(f"      Moderate:  ${p25:,.0f} - ${p50:,.0f}")
print(f"      High:      ${p50:,.0f} - ${p75:,.0f}")
print(f"      Very High: > ${p75:,.0f}")

# ------------------------------------------------------------------------------
# 8. COMBINED RISK SCORE
# ------------------------------------------------------------------------------

print(f"\n🎯 Calculating combined risk...")

max_cost = df_scen['Total_Cost_3mo'].max()
df_scen['Severity_Score'] = (df_scen['Total_Cost_3mo'] / max_cost) * 5.0

df_scen['Combined_Risk_Score'] = (
    df_scen['likelihood_error_score'] *
    df_scen['Severity_Score']
)

def combined_risk_band(score):
    if score < 5:
        return 'Low'
    elif score < 10:
        return 'Moderate'
    elif score < 15:
        return 'High'
    else:
        return 'Critical'

df_scen['Combined_Risk_Band'] = df_scen['Combined_Risk_Score'].apply(combined_risk_band)

print(f"\n   Distribution:")
for band in ['Low', 'Moderate', 'High', 'Critical']:
    count = (df_scen['Combined_Risk_Band'] == band).sum()
    pct = count / len(df_scen) * 100
    print(f"      {band:12}: {count:3} ({pct:5.1f}%)")

critical_underpaid = df_scen[
    (df_scen['Combined_Risk_Band'] == 'Critical') &
    (df_scen['Is_Underpaid'])
]

print(f"\n   ⚠️  CRITICAL UNDERPAYMENT: {len(critical_underpaid)} scenarios")

# ------------------------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------------------------

print(f"\n" + "=" * 80)
print("ENHANCED COST CALCULATION COMPLETE")
print("=" * 80)

print(f"\n✅ RESULTS:")
print(f"   {len(df_scen)} cost scenarios analyzed")
print(f"   {legal_count} high-value underpayment cases")
print(f"   {len(critical_underpaid)} critical + underpaid")

print(f"\n💰 COST SUMMARY:")
print(f"   3-month exposure: ${df_scen['Total_Cost_3mo'].sum():,.0f}")
print(f"   6-month exposure: ${df_scen['Total_Cost_6mo'].sum():,.0f}")

print(f"\n📊 KEY DATAFRAMES:")
print(f"   df_master: Original jobs ({len(df_master)} rows)")
print(f"   df_scen:   Cost scenarios ({len(df_scen)} rows)")

print("=" * 80)


COST ACCOUNTANT - JOB MISCLASSIFICATION COST ANALYZER v2.0

📋 Configuration:
   Base correction time: 24.0 hours
   HR loaded rate: $65.0/hour
   Benefit load factor: 1.3
   Cost windows: 3mo, 6mo
   Asymmetric risk: Underpay ×1.5, Overpay ×0.75
   Task Council Enabled: False

📂 Loading files...
   ✅ Master: 43 records
   ✅ Sample JDs: 43 records
   ✅ Salary: 68 roles

🔗 Linking Job Codes...
   ✅ Matched 43/43 records

💰 Creating salary map...
   ✅ Loaded 68 salary mappings
   📊 Range: $14,560 - $209,369

🔄 Expanding alternatives...
   ✅ Created 204 scenarios from 43 jobs
   📊 Avg alternatives/job: 4.7

💵 Calculating costs...

   📊 Underpaid: 87 (HIGH legal risk)
   📊 Overpaid:  117 (LOW risk)

   💰 With 30% benefits:
      Avg increase: $7,804

   ✅ Calculated costs for 183 valid scenarios

📊 Cost Statistics:
   3-Month Window:
      Average: $11,280
      Max:     $33,928
   6-Month Window:
      Average: $20,403
      Max:     $66,062

   ⚠️  Legal risk scenarios: 78

📊 Assigning se

# 7. Task Council

In [ ]:
# =========================================================================
# CHAT STEP 7: TASK COUNCIL INTEGRATION (optional but safe)
# - Purpose: if Council indicates true escalation, force worst-case hours to time_max
# =========================================================================

df_scen["Use Max Fix Time"] = False
df_scen["Qual Notes"] = ""

if USE_TASK_COUNCIL:
    # Minimal, deterministic integration:
    # If Council_Flag is true OR consensus is NO, treat as needing max-fix-time in worst-case scenario.
    # (You can later replace this with actual LLM calls if desired; this avoids drift/noise.)
    df_scen["Use Max Fix Time"] = df_scen.apply(
        lambda r: bool(r["Council_Flag"]) or (str(r["Consensus"]).upper() == "NO"),
        axis=1
    )
    df_scen.loc[df_scen["Use Max Fix Time"], "Qual Notes"] = "Escalated: Council_Flag or Consensus=NO"
    print("✅ Step 7 complete (deterministic council integration).")

    # Define time_max and operational_severity, and ensure columns exist if USE_TASK_COUNCIL is True
    # For this demonstration, we're assuming these would be defined/available if council is active.
    # However, since this block is currently for USE_TASK_COUNCIL = False, these lines won't execute.
    # If USE_TASK_COUNCIL was True, `time_max` would likely come from df_time[' Max'].iloc[0]
    # and `operational_severity` function would need to be defined.

    # Apply worst-case operational severity based on the override
    # This block needs 'Fin Sev', 'Dist Pen', 'Council_Flag' and operational_severity.
    # These are not present/defined in the current context unless council is fully set up.
    # For this fix, we are only addressing the 'Scenario Hours' Key Error if this code were to run.
    # If USE_TASK_COUNCIL is True, 'Scenario Hours' should be 'Hours'.
    # df_scen["WorstCase Hours"] = df_scen.apply(lambda r: (time_max if r["Use Max Fix Time"] else r["Hours"]), axis=1)
    # df_scen["WorstCase Ops Sev"] = df_scen.apply(
    #     lambda r: operational_severity(r["WorstCase Hours"], r["Consensus"], r["Council_Flag"]),
    #     axis=1
    # )
    #
    # df_scen["WorstCaseSeverity"] = df_scen["Fin Sev"] + df_scen["WorstCase Ops Sev"] + df_scen["Dist Pen"]
    # df_scen["WorstCaseBand"] = df_scen["WorstCaseSeverity"].apply(severity_band)

else:
    print("⏩ Task Council OFF: skipping council integration.")
    # The following lines are problematic when USE_TASK_COUNCIL is False because they rely on
    # undefined variables (time_max, operational_severity) and missing columns (Fin Sev, Dist Pen, Council_Flag).
    # They should only be executed if USE_TASK_COUNCIL is True and all prerequisites are met.
    # Therefore, they are commented out to prevent errors in this current state.

# Removed problematic lines that were outside the if/else block to prevent errors
# when USE_TASK_COUNCIL is False. If these calculations are desired when
# USE_TASK_COUNCIL is True, they should be moved inside the `if` block
# and all necessary variables/functions/columns must be properly defined/present.

display(df_scen.head(10))


⏩ Task Council OFF: skipping council integration.


,Job Code,Job Title,Predicted Role,Alternative Role,Consensus,human_error_probability,confusion_risk_score,similarity_numeric,likelihood_error_score,likelihood_band,...,Total_Cost_3mo,Pay_Impact_6mo,Total_Cost_6mo,Legal_Risk_Flag,Severity_Band,Severity_Score,Combined_Risk_Score,Combined_Risk_Band,Use Max Fix Time,Qual Notes
0,5003013,Department of Justice (DOJ) Grants Manager,Manager,Liaison,YES,24,2,46.2,3.66,High,...,8171.745563,12396.691125,14370.091125,False,Moderate,1.204281,4.407670,Low,False,
1,5003013,Department of Justice (DOJ) Grants Manager,Manager,Director,YES,24,2,46.2,3.66,High,...,23751.638625,43556.477250,45529.877250,True,Very High,3.500312,12.811141,High,False,
2,5003013,Department of Justice (DOJ) Grants Manager,Manager,Coordinator,YES,24,2,46.2,3.66,High,...,9785.650875,15624.501750,17597.901750,True,High,1.442125,5.278177,Moderate,False,
3,5003013,Department of Justice (DOJ) Grants Manager,Manager,Supervisor,YES,24,2,46.2,3.66,High,...,9202.003688,14457.207375,16430.607375,False,Moderate,1.356112,4.963370,Low,False,
4,5003013,Department of Justice (DOJ) Grants Manager,Manager,Administrative Assistant,YES,24,2,46.2,3.66,High,...,11745.944438,19545.088875,21518.488875,False,High,1.731016,6.335519,Moderate,False,
5,5501004,Contracting Agent,Analyst,Associate,YES,24,2,44.3,3.24,Moderate,...,14749.963313,25553.126625,27526.526625,False,Very High,2.173723,7.042861,Moderate,False,
6,5501004,Contracting Agent,Analyst,Coordinator,YES,24,2,44.3,3.24,Moderate,...,16420.506375,28894.212750,30867.612750,True,Very High,2.419913,7.840517,Moderate,False,
7,5501004,Contracting Agent,Analyst,Specialist,YES,24,2,44.3,3.24,Moderate,...,3885.487125,3824.174250,5797.574250,False,Low,0.572610,1.855255,Low,False,
8,5501004,Contracting Agent,Analyst,Auditor,YES,24,2,44.3,3.24,Moderate,...,2292.751500,638.703000,2612.103000,False,Low,0.337886,1.094751,Low,False,
9,1502020,Data Documentation and Management Analyst,Analyst,Associate,YES,24,2,44.3,3.24,Moderate,...,14749.963313,25553.126625,27526.526625,False,Very High,2.173723,7.042861,Moderate,False,


In [ ]:
# =========================================================================
# STEP 7: CONSOLIDATED AI AUDIT (Sonnet 4.5 + DeepSeek-R1)
# =========================================================================
if USE_TASK_COUNCIL:
    from anthropic import Anthropic
    from huggingface_hub import InferenceClient
    import json
    import re

    # 1. Initialize API Clients
    anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    hf_client = InferenceClient(token=HF_TOKEN)

    # 2. Define Sonnet 4.5 Qualitative Tagger
    def sonnet_qualitative_tag(pred_role, alt_role, salary_delta, consensus) -> dict:
        if not anthropic_client: return {"qual_band": "Moderate", "notes": "No Client"}
        prompt = f"""Assess the QUALITATIVE risk for misclassifying {pred_role} as {alt_role}.
        Salary delta is ${salary_delta:,.2f}. Consensus is {consensus}.
        Return ONLY a JSON object:
        {{"qual_band": "Low/Moderate/High/Critical", "risk_driver": [], "override": bool, "notes": "string"}}"""

        try:
            resp = anthropic_client.messages.create(
                model=PRIMARY_MODEL, max_tokens=1000, temperature=0,
                messages=[{"role": "user", "content": prompt}]
            )
            out = resp.content[0].text
            # Extract JSON if model wraps it in markdown
            m = re.search(r"\{.*\}", out, re.DOTALL)
            return json.loads(m.group(0)) if m else {"qual_band": "Moderate", "notes": "Parse Error"}
        except Exception as e:
            return {"qual_band": "Moderate", "notes": f"API Error: {str(e)}"}

    # 3. Define DeepSeek-R1 Operational Auditor
    def deepseek_audit_check(pred_role, alt_role, salary_delta) -> dict:
        if not hf_client: return {"use_max_fix_time": False}
        prompt = f"""<thought>Analyze if correcting {pred_role} to {alt_role} requires board approval or retroactive pay.</thought>
        Return ONLY a JSON object: {{"audit_flag": bool, "use_max_fix_time": bool, "notes": "string"}}"""

        try:
            response = hf_client.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                model=VALIDATOR_MODEL,
                temperature=0
            )
            out = response.choices[0].message.content
            m = re.search(r"\{.*\}", out, re.DOTALL)
            return json.loads(m.group(0)) if m else {"use_max_fix_time": False}
        except Exception:
            return {"use_max_fix_time": False}

    # 4. Execute the Audit
    tqdm.pandas()
    print("🤖 Running Tandem AI Audit (Sonnet 4.5 + DeepSeek-R1)...")

    # This runs the functions we just defined above
    df_scen["Sonnet"] = df_scen.progress_apply(
        lambda r: sonnet_qualitative_tag(r["Predicted Role"], r["Alternative Role"], r["Salary Delta"], r["Consensus"]),
        axis=1
    )
    df_scen["DeepSeek"] = df_scen.progress_apply(
        lambda r: deepseek_audit_check(r["Predicted Role"], r["Alternative Role"], r["Salary Delta"]),
        axis=1
    )

    # 5. Apply DeepSeek's "Worst-Case Time" Trigger
    df_scen["Use Max Fix Time"] = df_scen["DeepSeek"].apply(lambda d: bool(d.get("use_max_fix_time", False)))

    # Recalculate Final Operational Severity with the Council and R1 signals
    df_scen["Ops Sev (final)"] = df_scen.apply(
        lambda r: operational_severity(time_max if r["Use Max Fix Time"] else time_avg, r["Consensus"], r["Council_Flag"]),
        axis=1
    )

    # 6. Final Composite Severity Scores
    df_scen["WorstCaseSeverity"] = df_scen["Fin Sev"] + df_scen["Ops Sev (final)"] + df_scen["Dist Pen"]
    df_scen["WorstCaseBand"] = df_scen["WorstCaseSeverity"].apply(severity_band)

    print("✅ Step 7 Complete. AI Audit results integrated.")
else:
    print("⏩ Task Council is OFF. Skipping AI Audit.")

⏩ Task Council is OFF. Skipping AI Audit.


# **Scenario Analysis**

In [2]:
# ==============================================================================
# WORST-CASE SCENARIO ANALYSIS
# ==============================================================================
# For each job, identify the most expensive misclassification scenario
# ==============================================================================

print("=" * 80)
print("WORST-CASE SCENARIO ANALYSIS")
print("=" * 80)

# Find worst-case scenario per job (highest total cost)
worst_case_idx = df_scen.groupby('Job Code')['Total_Cost_3mo'].idxmax()
df_worst_case = df_scen.loc[worst_case_idx].copy()

# Sort by cost
df_worst_case = df_worst_case.sort_values('Total_Cost_3mo', ascending=False)

print(f"\n📊 Analyzed {len(df_worst_case)} jobs")
print(f"\n💰 Worst-Case Financial Exposure:")
print(f"   Total (3-month): ${df_worst_case['Total_Cost_3mo'].sum():,.0f}")
print(f"   Total (6-month): ${df_worst_case['Total_Cost_6mo'].sum():,.0f}")
print(f"   Average per job: ${df_worst_case['Total_Cost_3mo'].mean():,.0f}")

print(f"\n⚠️  High-Risk Cases:")
high_risk = df_worst_case[df_worst_case['Legal_Risk_Flag']]
print(f"   Underpayment risk: {len(high_risk)} jobs")
print(f"   Exposure: ${high_risk['Total_Cost_3mo'].sum():,.0f} - ${high_risk['Total_Cost_6mo'].sum():,.0f}")

print(f"\n🔥 Top 10 Highest-Cost Scenarios:")
top10 = df_worst_case.head(10)
for idx, row in top10.iterrows():
    print(f"\n   {row['Job Code']}: {row['Job Title'][:40]}")
    print(f"      {row['Predicted Role']} → {row['Alternative Role']}")
    print(f"      Cost: ${row['Total_Cost_3mo']:,.0f} - ${row['Total_Cost_6mo']:,.0f}")
    print(f"      Status: {row['Payment_Direction']}")
    if row['Legal_Risk_Flag']:
        print(f"      ⚠️  HIGH LEGAL RISK")

print("=" * 80)

WORST-CASE SCENARIO ANALYSIS

📊 Analyzed 40 jobs

💰 Worst-Case Financial Exposure:
   Total (3-month): $753,715
   Total (6-month): $1,427,816
   Average per job: $18,843

⚠️  High-Risk Cases:
   Underpayment risk: 29 jobs
   Exposure: $670,644 - $1,279,380

🔥 Top 10 Highest-Cost Scenarios:

   1901030: Director of Multi-Tiered System Support
      Director → Chief
      Cost: $33,928 - $66,062
      Status: UNDERPAID
      ⚠️  HIGH LEGAL RISK

   1901045: Director of Financial Operations
      Director → Chief
      Cost: $33,928 - $66,062
      Status: UNDERPAID
      ⚠️  HIGH LEGAL RISK

   1904004: Chief Financial Officer
      Director → Chief
      Cost: $33,928 - $66,062
      Status: UNDERPAID
      ⚠️  HIGH LEGAL RISK

   1904007: Chief of Student Support Services
      Director → Chief
      Cost: $33,928 - $66,062
      Status: UNDERPAID
      ⚠️  HIGH LEGAL RISK

   1901019: Director of Instructional Technology and
      Director → Chief
      Cost: $33,928 - $66,062
      

## 8. Export Results

In [5]:
# ==============================================================================
# ENHANCED EXPORT + RECONCILIATION - COMPLETE FIXED VERSION
# Implements colleague suggestions:
# 1. Exclusions & reconciliation (43 vs 40 jobs)
# 2. Two-layer reporting (scenario vs job-level)
# 3. Parameters export (reproducibility)
# 4. Top 10 drivers (executive briefing)
# 5. Locked definitions (audit defense)
# ==============================================================================

import os
import json
import pandas as pd
import numpy as np
from datetime import datetime
from google.colab import files

print("=" * 80)
print("ENHANCED EXPORT WITH RECONCILIATION")
print("=" * 80)

OUTDIR = "/content/output_cost_analysis"
os.makedirs(OUTDIR, exist_ok=True)

# ==============================================================================
# SECTION 1: RECONCILIATION - Track Excluded Jobs (FIXED)
# ==============================================================================

print("\n📊 RECONCILIATION: Tracking excluded jobs...")

# Ensure Job Code is string in both dataframes
df_master_copy = df_master.copy()
df_master_copy['Job Code'] = df_master_copy['Job Code'].astype(str).str.strip()

# Get input job list with ALL needed columns
required_cols = ['Job Code', 'job_title_original', 'major_role_group']
if 'alt_roles_list' in df_master_copy.columns:
    required_cols.append('alt_roles_list')

input_jobs = df_master_copy[required_cols].copy()

# Get successfully costed jobs
costed_jobs = set(df_scen['Job Code'].astype(str).str.strip().unique())

# Find excluded jobs
excluded_list = []
for idx, job_row in input_jobs.iterrows():
    job_code = str(job_row['Job Code']).strip()

    if job_code not in costed_jobs:
        # Determine exclusion reason
        predicted = job_row['major_role_group']

        # Get alt_list if column exists
        alt_list = job_row.get('alt_roles_list', None) if 'alt_roles_list' in job_row.index else None

        # Check various exclusion reasons
        if pd.isna(predicted) or str(predicted).strip() == '':
            reason = "Missing predicted role"
        elif predicted not in salary_map:
            reason = f"Missing salary data for predicted role '{predicted}'"
        elif alt_list is None or pd.isna(alt_list) or str(alt_list).strip() == '' or str(alt_list) == '[]':
            reason = "No alternative roles defined"
        else:
            # Check if alternatives have salary data
            try:
                from ast import literal_eval
                alts = literal_eval(str(alt_list))
                if not alts or len(alts) == 0:
                    reason = "No alternative roles defined"
                else:
                    missing_alts = [a for a in alts if str(a).strip() not in salary_map]
                    if len(missing_alts) == len(alts):
                        reason = f"Missing salary data for all alternatives"
                    elif missing_alts:
                        reason = f"Missing salary data for some alternatives: {', '.join(missing_alts[:2])}"
                    else:
                        reason = "Filtered during scenario validation (other reasons)"
            except Exception as e:
                reason = f"Invalid alternative roles format"

        excluded_list.append({
            'Job Code': job_code,
            'Job Title': job_row['job_title_original'],
            'Predicted Role': predicted,
            'Exclusion Reason': reason
        })

df_excluded = pd.DataFrame(excluded_list)

# Export exclusions
if len(df_excluded) > 0:
    exclusions_path = os.path.join(OUTDIR, "Excluded_Jobs.csv")
    df_excluded.to_csv(exclusions_path, index=False)
    print(f"   ⚠️  {len(df_excluded)} jobs excluded")
    print(f"   📄 Exported: Excluded_Jobs.csv")

    # Show summary of reasons
    print(f"\n   Exclusion breakdown:")
    reason_counts = df_excluded['Exclusion Reason'].value_counts()
    for reason, count in reason_counts.items():
        print(f"      • {reason}: {count}")

    # Show excluded jobs
    print(f"\n   Excluded jobs:")
    for idx, row in df_excluded.iterrows():
        print(f"      • {row['Job Code']}: {row['Job Title'][:50]}")
        print(f"        Predicted: {row['Predicted Role']}")
        print(f"        Reason: {row['Exclusion Reason']}")
else:
    print(f"   ✅ All {len(input_jobs)} jobs successfully costed")

# ==============================================================================
# SECTION 2: EXPORT MAIN FILES
# ==============================================================================

print("\n📁 Exporting main analysis files...")

# 2.1 - Full scenario data
full_cols = [
    'Job Code', 'Job Title',
    'Predicted Role', 'Alternative Role',
    'Payment_Direction', 'Is_Underpaid', 'Legal_Risk_Flag',
    'Salary_Delta_Abs', 'Total_Comp_Delta',
    'Hours', 'Risk_Multiplier',
    'Pay_Impact_3mo', 'Process_Cost', 'Total_Cost_3mo',
    'Pay_Impact_6mo', 'Total_Cost_6mo',
    'likelihood_error_score', 'likelihood_band',
    'Severity_Score', 'Severity_Band',
    'Combined_Risk_Score', 'Combined_Risk_Band',
    'Consensus', 'human_error_probability', 'confusion_risk_score',
    'similarity_numeric', 'review_priority', 'alt_count'
]

available_cols = [c for c in full_cols if c in df_scen.columns]
df_export_full = df_scen[available_cols].copy()

full_path = os.path.join(OUTDIR, "All_Cost_Scenarios.csv")
df_export_full.to_csv(full_path, index=False)
print(f"   ✅ All_Cost_Scenarios.csv ({len(df_export_full)} scenarios)")

# 2.2 - Worst-case per job
worst_case_idx = df_scen.groupby('Job Code')['Total_Cost_3mo'].idxmax()
df_worst_case = df_scen.loc[worst_case_idx].copy()
df_worst_case = df_worst_case.sort_values('Total_Cost_3mo', ascending=False)

worst_cols = [
    'Job Code', 'Job Title',
    'Predicted Role', 'Alternative Role',
    'Payment_Direction', 'Legal_Risk_Flag',
    'Total_Cost_3mo', 'Total_Cost_6mo',
    'likelihood_band', 'Severity_Band', 'Combined_Risk_Band'
]

available_worst = [c for c in worst_cols if c in df_worst_case.columns]
df_export_worst = df_worst_case[available_worst].copy()

worst_path = os.path.join(OUTDIR, "Worst_Case_Per_Job.csv")
df_export_worst.to_csv(worst_path, index=False)
print(f"   ✅ Worst_Case_Per_Job.csv ({len(df_export_worst)} jobs)")

# 2.3 - Critical underpayment (urgent)
critical_underpaid = df_scen[
    (df_scen['Combined_Risk_Band'] == 'Critical') &
    (df_scen['Is_Underpaid'] == True)
].copy()

critical_underpaid = critical_underpaid.sort_values('Total_Cost_3mo', ascending=False)

urgent_path = os.path.join(OUTDIR, "URGENT_Critical_Underpayment.csv")
critical_underpaid[available_cols].to_csv(urgent_path, index=False)
print(f"   ⚠️  URGENT_Critical_Underpayment.csv ({len(critical_underpaid)} scenarios)")

# ==============================================================================
# SECTION 3: EXPORT PARAMETERS (Reproducibility)
# ==============================================================================

print("\n⚙️  Exporting parameters for reproducibility...")

# Calculate thresholds from data
p25 = df_scen['Total_Cost_3mo'].quantile(0.25)
p50 = df_scen['Total_Cost_3mo'].quantile(0.50)
p75 = df_scen['Total_Cost_3mo'].quantile(0.75)

params = {
    "run_metadata": {
        "run_date": datetime.now().isoformat(),
        "notebook_version": "v2.0_VALIDATED",
        "total_jobs_input": len(input_jobs),
        "total_jobs_costed": len(df_worst_case),
        "total_jobs_excluded": len(df_excluded),
        "total_scenarios": len(df_scen)
    },
    "cost_assumptions": {
        "benefit_load_factor": float(BENEFIT_LOAD_FACTOR),
        "benefit_percentage": f"{int((BENEFIT_LOAD_FACTOR-1)*100)}%",
        "base_correction_hours": float(BASE_HOURS),
        "hr_loaded_rate_per_hour": float(HR_LOADED_RATE),
        "underpayment_risk_multiplier": float(UNDERPAYMENT_MULTIPLIER),
        "overpayment_risk_multiplier": float(OVERPAYMENT_MULTIPLIER),
        "correction_window_months_short": int(CORRECTION_WINDOW_MONTHS_SHORT),
        "correction_window_months_long": int(CORRECTION_WINDOW_MONTHS_LONG)
    },
    "severity_thresholds_3mo": {
        "low_max": float(p25),
        "moderate_max": float(p50),
        "high_max": float(p75),
        "very_high_min": float(p75),
        "method": "data-driven (percentiles: 25th, 50th, 75th)"
    },
    "definitions": {
        "Legal_Risk_Flag": "Is_Underpaid = True AND Total_Cost_3mo >= $5,000",
        "High_value_underpayment": "Same as Legal_Risk_Flag",
        "Critical_underpayment": "Combined_Risk_Band = 'Critical' AND Is_Underpaid = True",
        "Combined_Risk_Score": "likelihood_error_score × Severity_Score (0-25 scale)",
        "Severity_Score": "(Total_Cost_3mo / max_cost) × 5"
    }
}

params_path = os.path.join(OUTDIR, "cost_parameters.json")
with open(params_path, 'w') as f:
    json.dump(params, f, indent=2)

print(f"   ✅ cost_parameters.json")

# ==============================================================================
# SECTION 4: ENHANCED SUMMARY STATISTICS
# ==============================================================================

print("\n📊 Generating enhanced summary statistics...")

summary_path = os.path.join(OUTDIR, "Cost_Summary_Statistics.txt")

with open(summary_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("MISCLASSIFICATION COST ANALYSIS - ENHANCED SUMMARY\n")
    f.write("=" * 80 + "\n\n")

    # --- SECTION A: RECONCILIATION ---
    f.write("RECONCILIATION\n")
    f.write("-" * 80 + "\n")
    f.write(f"Total Jobs in Input (Master_Job_Analysis):     {len(input_jobs)}\n")
    f.write(f"Jobs Successfully Costed:                      {len(df_worst_case)}\n")
    f.write(f"Jobs Excluded (see Excluded_Jobs.csv):         {len(df_excluded)}\n")

    if len(df_excluded) > 0:
        f.write(f"\nExclusion Reasons:\n")
        for reason, count in df_excluded['Exclusion Reason'].value_counts().items():
            f.write(f"  - {reason}: {count}\n")

    f.write(f"\n")

    # --- SECTION B: SCENARIO-LEVEL ANALYSIS ---
    f.write("SCENARIO-LEVEL ANALYSIS\n")
    f.write("-" * 80 + "\n")
    f.write(f"Total Cost Scenarios Generated:                {len(df_scen)}\n")
    f.write(f"Average Alternatives per Job:                  {len(df_scen)/len(df_worst_case):.1f}\n\n")

    f.write("Risk Distribution (scenario-level):\n")
    for band in ['Low', 'Moderate', 'High', 'Critical']:
        count = (df_scen['Combined_Risk_Band'] == band).sum()
        pct = count / len(df_scen) * 100
        f.write(f"  {band:12}: {count:3} ({pct:5.1f}%)\n")

    f.write(f"\n")

    # --- SECTION C: JOB-LEVEL WORST-CASE EXPOSURE ---
    f.write("JOB-LEVEL WORST-CASE EXPOSURE\n")
    f.write("-" * 80 + "\n")
    f.write(f"Based on maximum cost scenario per job ({len(df_worst_case)} jobs):\n\n")

    f.write("Financial Exposure:\n")
    f.write(f"  3-Month Window Total:  ${df_worst_case['Total_Cost_3mo'].sum():,.2f}\n")
    f.write(f"  6-Month Window Total:  ${df_worst_case['Total_Cost_6mo'].sum():,.2f}\n")
    f.write(f"  Average Cost per Job:  ${df_worst_case['Total_Cost_3mo'].mean():,.2f} - ${df_worst_case['Total_Cost_6mo'].mean():,.2f}\n")
    f.write(f"  Median Cost per Job:   ${df_worst_case['Total_Cost_3mo'].median():,.2f} - ${df_worst_case['Total_Cost_6mo'].median():,.2f}\n")

    f.write(f"\n")

    # --- SECTION D: LEGAL RISK SCENARIOS ---
    f.write("LEGAL RISK SCENARIOS\n")
    f.write("-" * 80 + "\n")

    underpaid_count = df_scen['Is_Underpaid'].sum()
    legal_risk_count = df_scen['Legal_Risk_Flag'].sum()
    critical_underpaid_count = len(critical_underpaid)

    f.write(f"Underpayment Scenarios (all):                  {underpaid_count}\n")
    f.write(f"High-Value Underpayment (Legal_Risk_Flag):     {legal_risk_count}\n")
    f.write(f"Critical + Underpaid (URGENT):                 {critical_underpaid_count}\n")

    f.write(f"\n")

    # --- SECTION E: TOP 10 DRIVERS (Executive Briefing) ---
    f.write("TOP 10 COST DRIVERS - EXECUTIVE BRIEFING\n")
    f.write("-" * 80 + "\n")

    f.write("\nTop 10 Scenarios by 3-Month Cost:\n")
    top10_3mo = df_scen.nlargest(10, 'Total_Cost_3mo')
    for i, (idx, row) in enumerate(top10_3mo.iterrows(), 1):
        job_title = row['Job Title'][:40] if len(row['Job Title']) > 40 else row['Job Title']
        f.write(f"  {i:2}. {row['Job Code']}: {job_title}\n")
        f.write(f"      {row['Predicted Role']} → {row['Alternative Role']}\n")
        f.write(f"      ${row['Total_Cost_3mo']:,.0f} | {row['Combined_Risk_Band']}")
        if row['Is_Underpaid']:
            f.write(" | ⚠️  UNDERPAID")
        f.write("\n")

    f.write("\nTop 10 Scenarios by 6-Month Cost:\n")
    top10_6mo = df_scen.nlargest(10, 'Total_Cost_6mo')
    for i, (idx, row) in enumerate(top10_6mo.iterrows(), 1):
        job_title = row['Job Title'][:40] if len(row['Job Title']) > 40 else row['Job Title']
        f.write(f"  {i:2}. {row['Job Code']}: {job_title}\n")
        f.write(f"      {row['Predicted Role']} → {row['Alternative Role']}\n")
        f.write(f"      ${row['Total_Cost_6mo']:,.0f} | {row['Combined_Risk_Band']}")
        if row['Is_Underpaid']:
            f.write(" | ⚠️  UNDERPAID")
        f.write("\n")

    f.write("\nTop 10 Underpayment Risks (3-Month):\n")
    underpaid_scen = df_scen[df_scen['Is_Underpaid'] == True]
    top10_underpaid = underpaid_scen.nlargest(10, 'Total_Cost_3mo')
    for i, (idx, row) in enumerate(top10_underpaid.iterrows(), 1):
        job_title = row['Job Title'][:40] if len(row['Job Title']) > 40 else row['Job Title']
        f.write(f"  {i:2}. {row['Job Code']}: {job_title}\n")
        f.write(f"      {row['Predicted Role']} → {row['Alternative Role']}\n")
        f.write(f"      ${row['Total_Cost_3mo']:,.0f} | {row['Combined_Risk_Band']}\n")

    f.write(f"\n")

    # --- SECTION F: SEVERITY THRESHOLDS ---
    f.write("SEVERITY THRESHOLDS (3-month costs)\n")
    f.write("-" * 80 + "\n")
    f.write("Data-driven (percentile-based):\n")
    f.write(f"  Low:       < ${p25:,.2f} (bottom 25%)\n")
    f.write(f"  Moderate:  ${p25:,.2f} - ${p50:,.2f} (25th-50th percentile)\n")
    f.write(f"  High:      ${p50:,.2f} - ${p75:,.2f} (50th-75th percentile)\n")
    f.write(f"  Very High: > ${p75:,.2f} (top 25%)\n")

    f.write(f"\n")

    # --- SECTION G: METHODOLOGY & DEFINITIONS ---
    f.write("METHODOLOGY & DEFINITIONS\n")
    f.write("-" * 80 + "\n")

    f.write("Cost Calculation:\n")
    f.write(f"  Benefit load factor:         {BENEFIT_LOAD_FACTOR} ({int((BENEFIT_LOAD_FACTOR-1)*100)}% for total compensation)\n")
    f.write(f"  Base correction time:        {BASE_HOURS} hours (validated with MNPS)\n")
    f.write(f"  HR labor rate:               ${HR_LOADED_RATE}/hour (fully loaded)\n")
    f.write(f"  Asymmetric risk:             Underpay ×{UNDERPAYMENT_MULTIPLIER}, Overpay ×{OVERPAYMENT_MULTIPLIER}\n")
    f.write(f"  Cost windows:                {CORRECTION_WINDOW_MONTHS_SHORT} months, {CORRECTION_WINDOW_MONTHS_LONG} months\n\n")

    f.write("Key Definitions:\n")
    f.write("  Legal_Risk_Flag:\n")
    f.write("    Criteria: Is_Underpaid = True AND Total_Cost_3mo >= $5,000\n")
    f.write("    Rationale: High-value underpayment triggers wage claim risk\n")
    f.write(f"    Count: {legal_risk_count} scenarios\n\n")

    f.write("  High-value underpayment:\n")
    f.write("    Same as Legal_Risk_Flag\n")
    f.write(f"    Count: {legal_risk_count} scenarios\n\n")

    f.write("  Critical underpayment (URGENT):\n")
    f.write("    Criteria: Combined_Risk_Band = 'Critical' AND Is_Underpaid = True\n")
    f.write("    Rationale: High probability + High cost + Legal exposure\n")
    f.write(f"    Count: {critical_underpaid_count} scenarios\n\n")

    f.write("  Combined_Risk_Score:\n")
    f.write("    Formula: likelihood_error_score × Severity_Score\n")
    f.write("    Scale: 0-25 (likelihood 0-5 × severity 0-5)\n")
    f.write("    Bands: Low <5, Moderate 5-10, High 10-15, Critical >15\n\n")

    f.write("  Severity_Score:\n")
    f.write("    Formula: (Total_Cost_3mo / max_cost) × 5\n")
    f.write("    Normalizes cost to 0-5 scale for combination with likelihood\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("END OF REPORT\n")
    f.write("=" * 80 + "\n")

print(f"   ✅ Cost_Summary_Statistics.txt (ENHANCED)")

# ==============================================================================
# SECTION 5: DOWNLOAD FILES
# ==============================================================================

print("\n📥 Downloading files...")

files.download(full_path)
files.download(worst_path)
files.download(urgent_path)
files.download(params_path)
files.download(summary_path)

if len(df_excluded) > 0:
    files.download(exclusions_path)

print("\n" + "=" * 80)
print("EXPORT COMPLETE - ALL ENHANCEMENTS APPLIED")
print("=" * 80)

print(f"\n✅ Files exported to: {OUTDIR}")
print(f"\n   Core Files:")
print(f"   1. All_Cost_Scenarios.csv ({len(df_export_full)} scenarios)")
print(f"   2. Worst_Case_Per_Job.csv ({len(df_export_worst)} jobs)")
print(f"   3. URGENT_Critical_Underpayment.csv ({len(critical_underpaid)} urgent)")
print(f"\n   Enhancement Files:")
print(f"   4. Cost_Summary_Statistics.txt (ENHANCED with all sections)")
print(f"   5. cost_parameters.json (reproducibility)")

if len(df_excluded) > 0:
    print(f"   6. Excluded_Jobs.csv ({len(df_excluded)} excluded jobs)")

print(f"\n📊 Summary:")
print(f"   Input jobs:        {len(input_jobs)}")
print(f"   Successfully cost: {len(df_worst_case)}")
print(f"   Excluded:          {len(df_excluded)}")
print(f"   Total scenarios:   {len(df_scen)}")
print(f"   Critical urgent:   {len(critical_underpaid)}")

print("=" * 80)

ENHANCED EXPORT WITH RECONCILIATION

📊 RECONCILIATION: Tracking excluded jobs...
   ⚠️  3 jobs excluded
   📄 Exported: Excluded_Jobs.csv

   Exclusion breakdown:
      • Missing salary data for predicted role 'Assistant': 1
      • Missing salary data for predicted role 'Paraprofessional': 1
      • Missing salary data for predicted role 'Assistant Principal': 1

   Excluded jobs:
      • 8502009: Assistant Physical Therapist
        Predicted: Assistant
        Reason: Missing salary data for predicted role 'Assistant'
      • 1001003: Classroom Aide
        Predicted: Paraprofessional
        Reason: Missing salary data for predicted role 'Paraprofessional'
      • 6704006: Elementary School Assistant Principal
        Predicted: Assistant Principal
        Reason: Missing salary data for predicted role 'Assistant Principal'

📁 Exporting main analysis files...
   ✅ All_Cost_Scenarios.csv (183 scenarios)
   ✅ Worst_Case_Per_Job.csv (40 jobs)
   ⚠️  URGENT_Critical_Underpayment.csv (13

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


EXPORT COMPLETE - ALL ENHANCEMENTS APPLIED

✅ Files exported to: /content/output_cost_analysis

   Core Files:
   1. All_Cost_Scenarios.csv (183 scenarios)
   2. Worst_Case_Per_Job.csv (40 jobs)
   3. URGENT_Critical_Underpayment.csv (13 urgent)

   Enhancement Files:
   4. Cost_Summary_Statistics.txt (ENHANCED with all sections)
   5. cost_parameters.json (reproducibility)
   6. Excluded_Jobs.csv (3 excluded jobs)

📊 Summary:
   Input jobs:        43
   Successfully cost: 40
   Excluded:          3
   Total scenarios:   183
   Critical urgent:   13
